# citkid pipeline — overview and usage

This notebook walks through the `citkid.pipeline` framework end-to-end.

---

## Concepts

The pipeline has three layers:

| Layer | Class / function | Purpose |
|---|---|---|
| **Step definition** | `plStep` | Describes one unit of computation |
| **Data + calibration** | `DataSet` | Lazy zarr-backed parameter store + calibration pipeline execution |
| **Analysis execution** | `AnalysisRunner` | Runs analysis on a `DataSet` to create calibration parameters |
| **Interactive review** | `run_iq_analysis`, `run_ts_analysis`, … | Qt windows for per-resonator review |

**Data flow**

```
custom_steps.py
   load_global_data()        ─┐
   load_global_res_data()    ─┤  calibration pipeline (cal.yaml)
   load_data_f(data_idx)     ─┤  → DataSet caches & zarr store
   load_data_g(data_idx)     ─┘
                                     │
                             AnalysisRunner (iq_analysis.yaml)
                                fit_gain → fit_iq → …  →  zarr results
```

**`data_idx`** is the row index — one entry per (resonator, temperature/time
point).  
**`nrows`** is the total number of rows.

---

## func_type values for `plStep`

| `func_type` | Called once or per-row? | Produces |
|---|---|---|
| `'global'` | once | scalar / array used by all rows |
| `'global-res'` | once | per-resonator arrays (one value per row) |
| `'per-row'` | once per `data_idx` | per-row arrays, independent |
| `'vectorized'` | once per *group* of rows that share dependencies | per-row arrays, batched |

---

## YAML aliases

Pass these strings instead of a file path and the built-in template is used:

| alias | template file |
|---|---|
| `'iq'` | `iq_analysis.yaml` (fit_gain + fit_iq) |
| `'ts'` | `ts_analysis.yaml` (on-resonance timestream) |
| `'ts_offres'` | `ts_offres_analysis.yaml` (off-resonance gain) |

## 1. Defining custom steps with `plStep`

Write a `custom_steps.py` (or `custom_cal_steps` list) that loads your raw
data.  The four functions below are the standard interface.

The step list is the bridge between your data format and the pipeline — once
it's right, everything else is automatic.

In [ ]:
import numpy as np
import zarr
from citkid.pipeline.framework import plStep

# ---------------------------------------------------------------------------
# Replace the bodies of these four functions with your own data loading code.
# ---------------------------------------------------------------------------

def load_global_data():
    """Load data that is shared across all rows and all resonators.

    Must return: fres_all, qres_all, nrows
        fres_all  — 1-D array of resonator design frequencies (Hz), sorted.
        qres_all  — 1-D array of design Q values (same length).
        nrows     — total number of rows (int).
    """
    raw_root = zarr.open('path/to/raw.zarr', mode='r')
    nrows = int(raw_root['fres'].shape[0])

    fres_all = np.sort(np.load('path/to/fres_init.npy'))
    qres_all = np.ones_like(fres_all) * 8000
    return fres_all, qres_all, nrows


def load_global_res_data():
    """Load per-resonator data that is computed once (not per row).

    Must return: fres, qres, ares, res_idxs
        fres      — (nrows,) fitted resonator frequencies.
        qres      — (nrows,) fitted Q values.
        ares      — (nrows,) fitted amplitudes.
        res_idxs  — (nrows,) index mapping row → resonator in fres_all.
                    Use -1 for rows with no matched resonator.
    """
    root = zarr.open('path/to/raw.zarr', mode='r')
    fres     = np.array(root['fres'])
    ares     = np.array(root['ares'])
    qres     = np.array(root['qres'])
    res_idxs = np.array(root['res_idxs'])
    return fres, qres, ares, res_idxs


def load_data_f(data_idx):
    """Load the fine (resonance) IQ sweep for one row.

    data_idx — scalar int supplied by the pipeline.
    Must return: ff, zf  (both sorted on ff)
        ff  — 1-D frequency array (Hz).
        zf  — 1-D complex S21 array, same length.
    """
    root = zarr.open('path/to/fine_sweep.zarr', mode='r')
    ff = np.array(root['f'][data_idx, :])
    zf = np.array(root['z'][data_idx, :])
    idx = np.argsort(ff)
    return ff[idx], zf[idx]


def load_data_g(data_idx):
    """Load the gain (wide-band) IQ sweep for one row.

    Same return convention as load_data_f.
    """
    root = zarr.open('path/to/gain_sweep.zarr', mode='r')
    fg = np.array(root['f'][data_idx, :])
    zg = np.array(root['z'][data_idx, :])
    idx = np.argsort(fg)
    return fg[idx], zg[idx]


# ---------------------------------------------------------------------------
# Build the step list.  The tuples are:
#   (name, function, param_names, return_names, func_type)
#
# 'data_idx' in param_names tells the pipeline to pass the current row index.
# return_names become DataSet attribute names (e.g. DS.ff, DS.fres).
# ---------------------------------------------------------------------------
custom_cal_steps = [
    plStep('load_global_data',
           load_global_data,
           param_names=[],
           return_names=['fres_all', 'qres_all', 'nrows'],
           func_type='global'),

    plStep('load_global_res_data',
           load_global_res_data,
           param_names=[],
           return_names=['fres', 'qres', 'ares', 'res_idxs'],
           func_type='global-res'),

    plStep('load_data_f',
           load_data_f,
           param_names=['data_idx'],
           return_names=['ff', 'zf'],
           func_type='per-row'),

    plStep('load_data_g',
           load_data_g,
           param_names=['data_idx'],
           return_names=['fg', 'zg'],
           func_type='per-row'),
]

**Alternatively**, put the same content in a file (`my_custom_steps.py`) and
pass `custom_path='my_custom_steps.py'` to `DataSet`.  The file must expose a
module-level list called `custom_steps` (list of `plStep` objects).

## 2. DataSet — lazy zarr-backed parameter store

A `DataSet` wraps your zarr file and the calibration pipeline.  Attribute
access is lazy: `DS.ff` doesn't load anything until you index it with
`DS.ff[0]`.  Results computed by the pipeline are cached in memory and
optionally persisted to zarr.

In [ ]:
from citkid.pipeline.dataset import DataSet

zarr_path = 'path/to/output.zarr'   # created if it doesn't exist

DS = DataSet(
    zarr_path          = zarr_path,
    cal_yaml_path      = 'iq',               # alias → built-in iq cal YAML
    custom_cal_steps   = custom_cal_steps,   # list of plStep built above
    # custom_path      = 'my_custom_steps.py',  # or pass a file path instead
    zarr_mode          = 'a',                # 'r' to prevent accidental overwrite
)

### 2a. Accessing parameters

```python
DS.nrows           # global int — runs load_global_data once, caches result
DS.fres            # LazyAttrCollection (per-resonator array)
DS.fres[0]         # single row → scalar / array
DS.fres[[0,1,2]]   # multiple rows → stacked array
DS.ff[0]           # triggers load_data_f(0)
```

The most-recent run is always returned.  If data was saved to zarr in a
previous session it is read from disk; otherwise the pipeline step is run.

In [ ]:
print('nrows:', DS.nrows)
print('fres[0]:', DS.fres[0])

# Access raw data for a single resonator
ff0 = DS.ff[0]   # loads row 0 via load_data_f
zf0 = DS.zf[0]

# Access multiple rows at once
fres_block = DS.fres[[0, 1, 2]]   # shape (3,)

# The underlying zarr group is always available
print(DS.root.tree())

### 2b. Inspecting the zarr store

In [ ]:
# Show the zarr file in your file browser
DS.show_file('zarr')        # 'zarr' for the output store

# Check what produced a parameter and which run index it's from
print(DS.get_param_deps('ff', data_idx=0))

### 2c. Built-in plots

In [ ]:
import matplotlib.pyplot as plt

# Single plot type.  Available types depend on how far through the calibration
# pipeline you are: 'raw_data', 'gain_fit', 's21_rmv', 'circfit', 'sparper', 'xcal'
fig, axs = DS.plot(0, 'raw_data')
plt.show()

# Full calibration summary — all six panels combined into one PNG
# (requires that the full IQ calibration pipeline has already been run)
png_bytes = DS.plot_full_cal(0)
from IPython.display import Image; Image(png_bytes)

## 3. AnalysisRunner — execute the analysis pipeline

`AnalysisRunner` wraps a `DataSet` and runs a YAML-defined analysis.  It
handles user-adjustable parameters (e.g. masks, span multipliers) and can
save results to zarr.

In [ ]:
from citkid.pipeline.analysis import AnalysisRunner

AR = AnalysisRunner(
    DS,
    analysis_yaml_path = 'iq',    # alias → built-in IQ analysis YAML
    # custom_path = 'my_analysis_steps.py',  # optional custom steps
)

### 3a. Run the full pipeline on all rows

In [ ]:
# Run every step in iq_analysis.yaml for all rows, save results to zarr.
AR.execute_path(verbose=True)

### 3b. Run on a subset of rows

Use `np.where` to filter by any criterion — e.g. only on-resonance rows
(those where `res_idxs >= 0`).

In [ ]:
on_res_idxs = np.where(DS.res_idxs[:] >= 0)[0]
AR.execute_path(data_idx=on_res_idxs, verbose=True)

### 3c. Run a single step manually

`execute_step` lets you re-run one step with custom parameters without
re-running the whole pipeline.  Useful for tweaking a mask or span after an
interactive session.

In [ ]:
# Find the step object by name
fit_iq_step = next(s['task'] for s in AR.path if s['task'].name == 'fit_iq')

# Re-run fit_iq for row 5 with a custom mask, and save
custom_mask = np.ones(DS.ff[5].shape, dtype=bool)
custom_mask[:10] = False   # exclude first 10 points

AR.execute_step(
    fit_iq_step,
    data_idx = [5],
    user_params = {'iq_mask': custom_mask},
    save = True,
)

### 3d. Save in-memory results without re-running

During interactive sessions the pipeline accumulates results in memory but
does not write to disk until you ask.  Call `save_step_outputs` when you're
satisfied.

In [ ]:
# Execute without saving (the default), then save when ready
AR.execute_step(fit_iq_step, data_idx=[7], user_params={'iq_mask': None})

# --- review results here --- #

AR.save_step_outputs(fit_iq_step, data_idx=[7])

### 3e. Inspecting failures

When a per-row step fails for some rows, a warning is issued and the failure
tracebacks are stored.  They are also written into a `_failures` zarr group.

In [ ]:
# After execute_path or execute_step, inspect failures
if AR._last_failures:
    for data_idx, tb in AR._last_failures.items():
        print(f'--- data_idx {data_idx} ---')
        print(tb)

## 4. Interactive IQ analysis — `run_iq_analysis`

The interactive window lets you step through resonators (A / D or left/right
arrows), adjust the gain fit and IQ mask, and save results.

**Keyboard shortcuts**

| Key | Action |
|---|---|
| A / D or ← / → | Previous / next resonator |
| N | Run current panel, then auto-advance |
| Shift+N | Run current panel, stay put |
| R | Rescale plots |
| B | Mark current resonator as bad (`res_idxs = -1`) |

> **Note:** Run this cell from a terminal or a Qt-enabled Jupyter kernel.
> It will block until the window is closed.

In [ ]:
from citkid.pipeline.interactive import run_iq_analysis

run_iq_analysis(
    AR,
    start_idx  = 0,             # index into data_idxs to start at
    data_idxs  = on_res_idxs,   # restrict to on-resonance rows; None = all
    title      = 'IQ Analysis',
    ui_scale   = 1.0,           # scale text / chrome
    plot_scale = 1.0,           # scale plot area height
)

## 5. Interactive timestream analysis — `run_ts_analysis`

Same interface as `run_iq_analysis` but uses the `ts_analysis.yaml` pipeline.

The off-resonance (gain-only) variant is `run_gain_only_analysis`, which uses
`ts_offres_analysis.yaml` and shows only the gain panel.

In [ ]:
from citkid.pipeline.interactive import run_ts_analysis, run_gain_only_analysis
from citkid.pipeline.dataset import DataSet
from citkid.pipeline.analysis import AnalysisRunner

# --- on-resonance timestream ---
DS_ts = DataSet(
    zarr_path        = 'path/to/ts_output.zarr',
    cal_yaml_path    = 'ts',
    custom_cal_steps = custom_cal_steps,
)
AR_ts = AnalysisRunner(DS_ts, analysis_yaml_path='ts')

run_ts_analysis(
    AR_ts,
    start_idx  = 0,
    data_idxs  = None,   # None = all rows
    title      = 'Timestream Analysis',
)

# --- off-resonance / gain calibration ---
DS_offres = DataSet(
    zarr_path        = 'path/to/offres_output.zarr',
    cal_yaml_path    = 'ts_offres',
    custom_cal_steps = custom_cal_steps,
)
AR_offres = AnalysisRunner(DS_offres, analysis_yaml_path='ts_offres')

run_gain_only_analysis(
    AR_offres,
    start_idx  = 0,
    data_idxs  = None,
    title      = 'Off-Resonance Gain',
)

## 6. Sweep fitter — `run_sweep_fitter`

Use this when you have a *sweep over a parameter* (e.g. temperature, power)
and each value of the sweep parameter has its own zarr group (`sweep_000`,
`sweep_001`, …).  The window lets you navigate both resonators (A/D) and sweep
indices (W/S).

**Additional controls**

| Key / button | Action |
|---|---|
| W / S | Next / previous sweep index |
| Apply to All | Apply current panel settings to all sweep indices for this resonator |

You must supply a `make_custom_steps(sweep_idx)` factory that returns a fresh
step list for each sweep index (because each sweep index may point to a
different sub-group of the raw zarr file).

In [ ]:
import zarr
import numpy as np
from citkid.pipeline.framework import plStep
from citkid.pipeline.interactive import run_sweep_fitter

# Open the raw and output zarr groups
raw_root    = zarr.open('path/to/raw.zarr', mode='r')
output_root = zarr.open('path/to/sweep_output.zarr', mode='a')

n_sweep = 10       # number of sweep indices (sweep_000 … sweep_009)
nrows   = 50       # number of resonators per sweep

x_values = np.linspace(100, 400, n_sweep)  # e.g. temperatures in mK


def make_custom_steps(sweep_idx):
    """Return a fresh list of plStep objects for the given sweep index."""

    sweep_key = f'sweep_{sweep_idx:03d}'

    def load_global_data_sw():
        fres_all = np.sort(np.load('path/to/fres_init.npy'))
        qres_all = np.ones_like(fres_all) * 8000
        nrows_sw = int(raw_root[sweep_key]['fres'].shape[0])
        return fres_all, qres_all, nrows_sw

    def load_global_res_data_sw():
        grp = raw_root[sweep_key]
        return (np.array(grp['fres']), np.array(grp['qres']),
                np.array(grp['ares']), np.array(grp['res_idxs']))

    def load_data_f_sw(data_idx):
        grp = raw_root[sweep_key]['fine_sweep']
        ff = np.array(grp['f'][data_idx, :])
        zf = np.array(grp['z'][data_idx, :])
        idx = np.argsort(ff)
        return ff[idx], zf[idx]

    def load_data_g_sw(data_idx):
        grp = raw_root[sweep_key]['gain_sweep']
        fg = np.array(grp['f'][data_idx, :])
        zg = np.array(grp['z'][data_idx, :])
        idx = np.argsort(fg)
        return fg[idx], zg[idx]

    return [
        plStep('load_global_data',     load_global_data_sw,
               [], ['fres_all', 'qres_all', 'nrows'], 'global'),
        plStep('load_global_res_data', load_global_res_data_sw,
               [], ['fres', 'qres', 'ares', 'res_idxs'], 'global-res'),
        plStep('load_data_f',          load_data_f_sw,
               ['data_idx'], ['ff', 'zf'], 'per-row'),
        plStep('load_data_g',          load_data_g_sw,
               ['data_idx'], ['fg', 'zg'], 'per-row'),
    ]


run_sweep_fitter(
    make_custom_steps   = make_custom_steps,
    cal_yaml_path       = 'iq',
    analysis_yaml_path  = 'iq',
    root                = output_root,
    n_sweep             = n_sweep,
    x_param_name        = 'T_mc',   # name stored in DS after load; or use x_values
    x_name              = 'T (mK)', # axis label shown in the scatter panel
    y_param_name        = 'fres',   # y-axis parameter for the scatter panel
    y_name              = 'fres (Hz)',
    x_values            = x_values, # if you don't have x_param_name in DS
    title               = 'Sweep Fitter',
    ui_scale            = 1.0,
    plot_scale          = 1.0,
)

## 7. Accessing results after analysis

Once the analysis pipeline has run (interactively or in batch), results are
accessible as `DataSet` attributes.  They are backed by zarr and survive
session restarts — just re-open the same zarr file.

In [ ]:
# Access IQ fit results
iq_popt_0 = DS.iq_popt[0]          # single row
iq_popt_all = DS.iq_popt[:]        # all rows (stacked)

print('iq_popt shape:', iq_popt_all.shape)

# Resonator frequencies and Q values from the fit
fres_fit = DS.fres[:]              # shape (nrows,)
qres_fit = DS.qres[:]

In [ ]:
# Re-open a DataSet from a previous session — no recomputation needed
DS2 = DataSet(
    zarr_path        = zarr_path,
    cal_yaml_path    = 'iq',
    custom_cal_steps = custom_cal_steps,
    zarr_mode        = 'a',
)
# Zarr-backed results are available immediately
print(DS2.iq_popt[0])

## Tips and common patterns

**Run the pipeline in batch first, then do interactive review.**
Call `AR.execute_path(verbose=True)` to get default results for all rows,
then use `run_iq_analysis` to fix the small fraction that need manual
adjustment.  Interactive saves overwrite only the resonators you touch.

**Re-running a step increments the run index.**
Each call to `execute_step` (or the interactive N key) creates a new run
entry in zarr rather than overwriting the old one.  The most-recent run
is always returned by `DS.param[...]`.

**`save=False` (preview mode).**
The interactive UI runs steps with `save=False` by default so you can
preview the result before committing.  Press N (or click Apply / Apply to
All) to save.

**Filter to on-resonance rows.**
```python
on_res = np.where(DS.res_idxs[:] >= 0)[0]
AR.execute_path(data_idx=on_res)
```

**Use `show_file` to inspect what is stored.**
```python
DS.show_file('zarr')   # prints zarr tree with run groups
```